In [4]:
# from google.colab import drive
# drive.mount('/content/drive')

In [5]:
# !unzip "/content/drive/MyDrive/archive.zip" -d "./medicinal_leaf/"

In [6]:
import os
import random
import uuid
from PIL import Image
try:
    import torchvision.transforms.v2 as v2
    has_torchvision = True
except ImportError:
    has_torchvision = False
    print("Warning: torchvision not found. Proceeding with basic Pillow transforms only.")

if has_torchvision:
    augment_transform = v2.Compose([
        v2.RandomAffine(degrees=30, translate=(0.2, 0.2), scale=(0.8, 1.2), shear=11.45, interpolation=v2.InterpolationMode.NEAREST),
        v2.RandomHorizontalFlip(p=0.5)
    ])
else:
    def augment_transform(img):
        w, h = img.size
        if random.random() > 0.5:
            import PIL.ImageOps
            img = PIL.ImageOps.mirror(img)
        angle = random.uniform(-30, 30)
        img = img.rotate(angle, resample=Image.Resampling.NEAREST, translate=(random.uniform(-0.2,0.2)*w, random.uniform(-0.2,0.2)*h))
        return img

IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp')

def list_images(directory):
    return [
        f for f in os.listdir(directory)
        if f.lower().endswith(IMAGE_EXTENSIONS)
    ]

def augment_images_from_random_file(input_class_dir, output_class_dir, num_samples=5):
    os.makedirs(output_class_dir, exist_ok=True)
    image_files = [f for f in list_images(input_class_dir) if not f.startswith('aug_')]
    if not image_files:
        image_files = list_images(input_class_dir)
    if not image_files:
        return
    random_image = random.choice(image_files)
    img = Image.open(os.path.join(input_class_dir, random_image)).convert('RGB')
    for _ in range(num_samples):
        aug_img = augment_transform(img)
        out_filename = f"aug_{uuid.uuid4().hex[:8]}.jpeg"
        out_path = os.path.join(output_class_dir, out_filename)
        aug_img.save(out_path, format='JPEG', quality=95)
    print(f"Generated {num_samples} augmented images from {random_image}")

base_dir = './Medicinal_Leaves'
output_dir = './result'
target_image_count = 500
os.makedirs(output_dir, exist_ok=True)

for class_name in os.listdir(base_dir):
    class_dir = os.path.join(base_dir, class_name)
    if not os.path.isdir(class_dir):
        continue
    output_class_dir = os.path.join(output_dir, class_name)
    os.makedirs(output_class_dir, exist_ok=True)
    current_image_count = len(list_images(class_dir)) + len(list_images(output_class_dir))
    if current_image_count < target_image_count:
        images_needed = target_image_count - current_image_count
        while images_needed > 0:
            batch_size = min(5, images_needed)
            augment_images_from_random_file(class_dir, output_class_dir, num_samples=batch_size)
            current_image_count = len(list_images(class_dir)) + len(list_images(output_class_dir))
            images_needed = target_image_count - current_image_count
    print(f"{class_name}: {len(list_images(output_class_dir))} generated images in {output_class_dir}")

print("Augmentation completed!")


Generated 5 augmented images from 240.jpg
Generated 3 augmented images from 18.jpg
Lemon: 8 generated images in ./result/Lemon
Papaya: 0 generated images in ./result/Papaya
Generated 5 augmented images from 726.jpg
Generated 4 augmented images from 658.jpg
Mint: 9 generated images in ./result/Mint
Generated 1 augmented images from 346.jpg
Coriender: 1 generated images in ./result/Coriender
Generated 5 augmented images from bdfdfgf.jpg
Generated 4 augmented images from dsdffsfdsfd.jpg
Amla: 9 generated images in ./result/Amla
Generated 5 augmented images from P_20190921_122508.jpg
Generated 4 augmented images from IMG_20190909_134443.jpg
Bringaraja: 9 generated images in ./result/Bringaraja
Generated 1 augmented images from 10009_jpg.rf.f68877f4b97e51fe7c1fe7b0089b3297.jpg
Neem: 1 generated images in ./result/Neem
Generated 5 augmented images from 2266.jpg
Bhrami: 5 generated images in ./result/Bhrami
Generated 5 augmented images from 3.jpeg
Generated 5 augmented images from 20190911_11